# Final Project STA 4724: Stock Market PCA # 

## Data Aquisition & Processing ##
Aquire data from yfinance and arrange a covariance matrix of log returns

In [50]:
import yfinance as yfin
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [51]:
import pandas as pd
import yfinance as yf

storage_options = {
    'User-Agent': 'MyCoolDataScript/1.0 (contact@example.com) BasedOnPythonRequests'
}

# 1. Get the current list of S&P 500 companies
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
sp500_table = pd.read_html(url, storage_options=storage_options)[0]

tickers = sp500_table["Symbol"].tolist()

# Yahoo Finance uses BRK-B instead of BRK.B, for example
tickers = [ticker.replace(".", "-") for ticker in tickers]

# 2. Download monrhtly stock prices for 2021-2025
start_date = "2021-01-01"
end_date = "2026-01-01"  # End date is exclusive

data = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    interval="1mo",
    auto_adjust=False,
    group_by="column",
    progress=True,
    threads=True
)

# 3. Extract adjusted closing prices
prices = data["Adj Close"].copy()

# 4. Make sure the columns are ticker symbols
prices.columns.name = None
prices.index.name = "Month"

# 5. Sort by date
prices = prices.sort_index()

# # 6. Save the DataFrame to a CSV file
# prices.to_csv("sp500_monthly_prices_2021-25.csv")

# 7. Display the results
print("DataFrame shape:", prices.shape)
print("\nFirst five rows:")
print(prices.head())

print("\nMissing values per stock (top 20):")
print(prices.isna().sum().sort_values(ascending=False).head(20))

# print("\nSaved to sp500_daily_prices_2025.csv")

[*************         28%                       ]  140 of 503 completed$FDXF: possibly delisted; no price data found  (1mo 2021-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1767243600")
[**********************63%*****                  ]  318 of 503 completed$HONA: possibly delisted; no price data found  (1mo 2021-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1767243600")
[**********************79%*************          ]  396 of 503 completed$VYLR: possibly delisted; no price data found  (1mo 2021-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1767243600")
[*********************100%***********************]  503 of 503 completed

3 Failed downloads:
['FDXF', 'HONA', 'VYLR']: possibly delisted; no price data found  (1mo 2021-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1767243600")


DataFrame shape: (60, 503)

First five rows:
                     A        AAPL       ABBV        ABNB         ABT  \
Month                                                                   
2021-01-01  115.479385  128.108078  82.835785  183.630005  110.622246   
2021-02-01  117.305237  117.896317  87.087517  206.350006  107.212006   
2021-03-01  122.177353  118.761627  87.475533  187.940002  107.265739   
2021-04-01  128.619034  127.813370  91.222580  172.710007  107.875168   
2021-05-01  132.940369  121.359177  92.613434  140.399994  104.793785   

                 ACGL         ACN        ADBE         ADI        ADM  ...  \
Month                                                                 ...   
2021-01-01  29.867723  219.622406  458.769989  133.812439  42.626995  ...   
2021-02-01  34.061184  227.774658  459.670013  142.129776  48.564194  ...   
2021-03-01  36.485977  250.788223  475.369995  141.454727  48.924694  ...   
2021-04-01  37.760181  264.058319  508.339996  139.703476 

## Convert prices to log returns covariance matrix ##  
we will investigate a scaled correlation matrix and an unscaled covariance matrix

In [52]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Drop missing data from the original prices DataFrame
prices_clean = prices.dropna(axis=1, how="any")

# 2. Calculate Log Returns first (removes negative value risks from scaling)
#    We drop the first row because .diff() creates an initial row of NaNs
X_log_returns = np.log(prices_clean).diff().dropna()

# 3. Scale the log returns (PCA is highly sensitive to variance scales)
X_log_returns_scaled = StandardScaler().fit_transform(X_log_returns)

# 4. Fit and apply PCA
pca = PCA()
X_log_returns_pca = pca.fit_transform(X_log_returns_scaled)

X_log_returns_var_expl = pca.explained_variance_

# 5. Get the asset correlation matrix from the clean prices
X_log_returns_corr = prices_clean.corr()

## Eigenvalues vector and Eigenvector Matrix

In [53]:
# eigenvalues
pd.DataFrame(X_log_returns_var_expl).head(5)

,0
0,162.253701
1,35.766717
2,29.040206
3,15.665023
4,13.189424


In [54]:
# eigenvectors
pd.DataFrame(X_log_returns_pca).head(5)

,0,1,2,3,4,5,6,7,8,9,...,49,50,51,52,53,54,55,56,57,58
0,11.192726,-3.991301,17.639071,4.079606,-1.824238,3.446889,0.491385,-5.827905,1.313064,1.702806,...,-0.051845,0.708254,1.109251,0.647244,0.086776,0.025433,1.253067,0.283124,0.295934,-5.608614e-15
1,14.359527,10.943601,-4.555037,-2.579752,3.602183,-0.567888,-4.447969,0.180949,0.668882,-0.546234,...,-0.814964,-0.083850,1.067687,-1.045141,-0.136396,-0.498193,2.249617,-0.679627,-0.989845,-5.608614e-15
2,11.533466,2.659111,-2.974786,2.701744,-3.537175,0.001040,0.678108,-0.221108,-1.649840,-1.102730,...,2.057789,-1.632247,-0.898133,-1.563601,-1.122053,-2.302162,1.967897,-1.695104,-1.053822,-5.608614e-15
3,2.371452,0.873216,3.322136,-1.231558,-1.304824,0.286778,-0.588127,-0.156161,-1.811745,2.429880,...,1.956479,1.497052,-1.498824,-0.331652,-2.162454,-2.256166,-0.127913,-0.435000,1.812141,-5.608614e-15
4,-2.764486,-5.752850,-6.030067,1.585480,-2.268857,1.691573,5.007204,-2.149612,-3.179682,0.371255,...,1.738277,-4.227234,0.467980,1.320482,0.649140,-0.366690,3.566354,-2.664683,0.227864,-5.608614e-15


## Now to determine which principle directions to target ##  
find this by 

## Now to allocate assets with short distances from target principle directions ##
- find least SSE assets with target principle directions.
- for assets in p>2 eigenvectors, find low correlation assets to p=1 eigenvector.

In [55]:
X_log_returns_corr

,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
A,1.000000,0.017787,-0.048386,0.092999,0.125348,-0.123491,0.276499,0.275118,0.071911,-0.010900,...,0.165142,0.098220,0.022911,-0.154236,0.369840,0.376939,0.035325,0.215461,0.465068,0.349096
AAPL,0.017787,1.000000,0.897855,-0.232416,0.538168,0.837209,0.292177,-0.154416,0.906975,-0.315211,...,-0.362050,0.195806,0.589705,0.721724,0.681375,-0.516949,0.822797,-0.656999,-0.369053,-0.179349
ABBV,-0.048386,0.897855,1.000000,-0.395967,0.519927,0.870931,0.130627,-0.404466,0.886051,-0.204317,...,-0.412783,0.163486,0.637960,0.841133,0.640662,-0.660037,0.864621,-0.731114,-0.553908,-0.365722
ABNB,0.092999,-0.232416,-0.395967,1.000000,0.184051,-0.326298,0.162407,0.549819,-0.170773,-0.465244,...,0.296818,0.328929,-0.421855,-0.582189,0.046387,0.696160,-0.291029,0.525111,0.641755,0.380076
ABT,0.125348,0.538168,0.519927,0.184051,1.000000,0.296074,0.219844,-0.059248,0.534909,-0.578231,...,-0.218732,0.304480,0.610978,0.077127,0.591539,0.119901,0.610999,-0.267306,0.199003,0.034666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XYZ,0.376939,-0.516949,-0.660037,0.696160,0.119901,-0.694719,-0.058744,0.497182,-0.456524,-0.331420,...,0.186389,0.272644,-0.251631,-0.888064,-0.054620,1.000000,-0.513261,0.734921,0.883083,0.371554
YUM,0.035325,0.822797,0.864621,-0.291029,0.610999,0.830978,0.252481,-0.198697,0.863579,-0.308504,...,-0.347418,0.261671,0.527603,0.748817,0.744038,-0.513261,1.000000,-0.540492,-0.402127,-0.145411
ZBH,0.215461,-0.656999,-0.731114,0.525111,-0.267306,-0.612017,-0.122309,0.429826,-0.570251,-0.049196,...,0.344082,0.337493,-0.497268,-0.686576,-0.267333,0.734921,-0.540492,1.000000,0.550116,0.281315
ZBRA,0.465068,-0.369053,-0.553908,0.641755,0.199003,-0.586932,0.206669,0.528188,-0.351809,-0.306414,...,0.372197,0.104462,-0.196132,-0.783818,-0.001877,0.883083,-0.402127,0.550116,1.000000,0.583349


In [ ]:
#test

# What needs to be ironed out... #
- Is there a more meaningful timeframe of data to use?
- Possible to structure future principle component analysis as an infinitely rolling window with exponential decay of older entries?
- How to allocate funds to assets based on principle direction?